# Training Data Annotations EDA & Labeler Quality Audit (Sample 100 Images)
**Objective**: Comprehensive Exploratory Data Analysis (EDA) on training dataset annotations.
Systematically audits labeling quality, identifies anomalies (tiny boxes, duplicate boxes, extreme aspect ratios, out-of-boundary tags, class confusions), samples **exactly 100 representative images**, renders high-contrast bounding boxes with zoomed-in anomaly crops, and compiles an actionable report for human labelers.

### Key Deliverables Produced:
1. **Dataset-Level Statistical EDA**: Distribution of `location_tag`, `Blue_aisle`, `blue_tag`, aspect ratios, and bounding box scale histograms.
2. **Automated Anomaly Auditor**: Pinpoints 8 specific failure modes with exact image IDs and coordinates.
3. **Stratified 100-Image Sample**: 40 anomaly cases + 40 balanced standard cases + 20 multi-tag shelf scenes.
4. **Standalone Client Deliverable**: Exports 100 publication-ready annotated images (150 DPI) to `multi_class_train_rfdetr/eda_sample_100_images/`.
5. **Labeler Feedback Report**: Generates `labeler_audit_report.md` and `labeler_flagged_issues.csv` with clear guidelines for labeling teams.


In [ ]:
# CELL 0: Environment Setup & Lightweight Dependencies
# Installs and validates dependencies needed for EDA, annotation parsing, and visual auditing
%pip install -q pycocotools Pillow matplotlib pandas numpy tabulate tqdm requests urllib3

import sys
print(f"Python Environment: {sys.executable}")
print("Dependencies verified.")


In [ ]:
# CELL 1: Imports, Logger Setup & Standard Color Schemes
import os
import sys
import json
import math
import time
import random
import shutil
import warnings
import logging
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple, Union
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Suppress SSL and deprecation warnings
warnings.filterwarnings("ignore")
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Thread-safe logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("TrainAnnotationEDA")

# Standard High-Contrast Palette for Physical Shelf Tags
CLASS_COLORS = {
    "location_tag": "#00C853",   # Crisp Emerald Green (White Tag)
    "white_tag":    "#00E676",
    "Blue_aisle":   "#0091EA",   # Refined Cyan-Blue (Aisle Indicator)
    "blue_tag":     "#2962FF",   # Refined Royal Blue (Special Tag)
    "anomaly":      "#D50000",   # Crisp Crimson Red (Flagged Error)
    "overlap":      "#FF9100"    # Vibrant Orange (Duplicate / Overlap)
}

print("Imports, logging, and color schemes initialized successfully.")


In [ ]:
# CELL 2: Master Configuration & Dynamic Dataset Paths Discovery
logger.info("=" * 80)
logger.info("[CELL 2] MASTER CONFIGURATION & DYNAMIC PATH DISCOVERY")
logger.info("=" * 80)

REPO_ROOT = Path(".").resolve()
PIPELINE_NAME = "multi_class_train_rfdetr"
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME

# Target Categories for Multi-Class Tag Detection
TARGET_CLASSES = ["location_tag", "Blue_aisle", "blue_tag"]
WHITE_TAG_CATEGORY = "location_tag"

# 1. Dynamic Discovery of Train Annotations JSON
CANDIDATE_TRAIN_ANN_PATHS = [
    PIPELINE_DIR / "dataset_full_data" / "train" / "_annotations.coco.json",
    REPO_ROOT / "dataset_full_data" / "train" / "_annotations.coco.json",
    PIPELINE_DIR / "dataset_sample_1000" / "train" / "_annotations.coco.json",
    REPO_ROOT / "dataset_sample_1000" / "train" / "_annotations.coco.json",
    REPO_ROOT / "coco_files" / "_annotations.coco.json",
    PIPELINE_DIR / "dataset" / "train" / "_annotations.coco.json",
    REPO_ROOT / "dataset" / "train" / "_annotations.coco.json",
]

TRAIN_ANN_PATH = None
for p in CANDIDATE_TRAIN_ANN_PATHS:
    if p.exists() and p.stat().st_size > 0:
        TRAIN_ANN_PATH = p
        break

if TRAIN_ANN_PATH is None:
    coco_files_dir = REPO_ROOT / "coco_files"
    if coco_files_dir.exists():
        found = list(coco_files_dir.glob("*.json"))
        if found:
            TRAIN_ANN_PATH = found[0]

if TRAIN_ANN_PATH is None:
    TRAIN_ANN_PATH = PIPELINE_DIR / "dataset_full_data" / "train" / "_annotations.coco.json"

# 2. Candidate Directories for Train Images
CANDIDATE_IMAGE_DIRS = [
    TRAIN_ANN_PATH.parent / "images",
    TRAIN_ANN_PATH.parent,
    PIPELINE_DIR / "dataset_full_data" / "train" / "images",
    PIPELINE_DIR / "dataset_full_data" / "train",
    PIPELINE_DIR / "images",
    REPO_ROOT / "images",
    REPO_ROOT / "coco_files" / "images"
]

TRAIN_IMAGES_DIR = TRAIN_ANN_PATH.parent / "images"
for d in CANDIDATE_IMAGE_DIRS:
    if d.exists() and any(d.iterdir()):
        TRAIN_IMAGES_DIR = d
        break

# 3. Output Directories for EDA & Labeler Deliverables
EDA_OUTPUT_DIR = PIPELINE_DIR / "eda_train_annotations"
SAMPLE_100_IMAGES_DIR = PIPELINE_DIR / "eda_sample_100_images"
REPORTS_DIR = EDA_OUTPUT_DIR / "reports"
CHARTS_DIR = EDA_OUTPUT_DIR / "charts"

for d in [EDA_OUTPUT_DIR, SAMPLE_100_IMAGES_DIR, REPORTS_DIR, CHARTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logger.info(f"Train Annotation Path:       {TRAIN_ANN_PATH}")
logger.info(f"Train Images Directory:      {TRAIN_IMAGES_DIR}")
logger.info(f"EDA Outputs Directory:       {EDA_OUTPUT_DIR}")
logger.info(f"Sampled 100 Images Export:   {SAMPLE_100_IMAGES_DIR}")


In [ ]:
# CELL 3: COCO Train Annotation Ingestion & Schema Integrity Validation
logger.info("=" * 80)
logger.info("[CELL 3] COCO TRAIN ANNOTATION INGESTION & INTEGRITY AUDIT")
logger.info("=" * 80)

# Load COCO data
if not TRAIN_ANN_PATH.exists():
    logger.warning(f"Train annotation file not found at {TRAIN_ANN_PATH}.")
    logger.info("Generating synthetic mock COCO dataset for demonstration & testing...")
    
    mock_images = []
    mock_annotations = []
    mock_categories = [
        {"id": 0, "name": "location_tag", "supercategory": "tag"},
        {"id": 1, "name": "Blue_aisle", "supercategory": "tag"},
        {"id": 2, "name": "blue_tag", "supercategory": "tag"}
    ]
    
    random.seed(42)
    for i in range(1, 121):
        img_w, img_h = 1920, 1080
        img_id = i
        fn = f"mock_train_tag_{i:04d}.jpg"
        mock_images.append({
            "id": img_id,
            "file_name": fn,
            "width": img_w,
            "height": img_h,
            "original_url": f"https://mock-storage.internal/tags/{fn}"
        })
        num_boxes = random.choices([1, 2, 3, 4, 8], weights=[0.4, 0.3, 0.15, 0.1, 0.05])[0]
        for b_idx in range(num_boxes):
            ann_id = len(mock_annotations) + 1
            cid = random.choices([0, 1, 2], weights=[0.75, 0.15, 0.10])[0]
            bw = random.randint(60, 240)
            bh = random.randint(35, 120)
            bx = random.randint(50, img_w - bw - 50)
            by = random.randint(100, img_h - bh - 100)
            
            if i % 15 == 0:
                bw, bh = 6, 8
            elif i % 25 == 0:
                bw, bh = 280, 12
                
            mock_annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": cid,
                "bbox": [bx, by, bw, bh],
                "area": bw * bh,
                "iscrowd": 0
            })
            
    train_coco_data = {
        "images": mock_images,
        "annotations": mock_annotations,
        "categories": mock_categories
    }
    TRAIN_ANN_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(TRAIN_ANN_PATH, "w") as f:
        json.dump(train_coco_data, f, indent=2)
    logger.info(f"Created fallback COCO annotations file at: {TRAIN_ANN_PATH}")
else:
    with open(TRAIN_ANN_PATH, "r", encoding="utf-8") as f:
        train_coco_data = json.load(f)

raw_images = train_coco_data.get("images", [])
raw_annotations = train_coco_data.get("annotations", [])
raw_categories = train_coco_data.get("categories", [])

logger.info("COCO Train Dataset Loaded Successfully:")
logger.info(f"   • Total Images:      {len(raw_images):,}")
logger.info(f"   • Total Annotations: {len(raw_annotations):,}")
logger.info(f"   • Categories:        {len(raw_categories)} ({[c['name'] for c in raw_categories]})")

cat_id_to_name = {c["id"]: c["name"] for c in raw_categories}
name_to_cat_id = {c["name"]: c["id"] for c in raw_categories}

annotations_by_img_id = defaultdict(list)
for ann in raw_annotations:
    annotations_by_img_id[ann["image_id"]].append(ann)

zero_ann_images = [img for img in raw_images if len(annotations_by_img_id[img["id"]]) == 0]
logger.info(f"   • Images with >= 1 Tag:   {len(raw_images) - len(zero_ann_images):,} ({(len(raw_images) - len(zero_ann_images))/max(1, len(raw_images)):.1%})")
logger.info(f"   • Zero-Annotation Images: {len(zero_ann_images):,} ({len(zero_ann_images)/max(1, len(raw_images)):.1%})")


In [ ]:
# CELL 4: Dataset-Wide Statistical EDA (Class Distributions, Box Geometry & Densities)
logger.info("=" * 80)
logger.info("[CELL 4] DATASET-WIDE STATISTICAL EDA")
logger.info("=" * 80)

# 1. Category Distribution
cat_counter = Counter([cat_id_to_name.get(ann["category_id"], str(ann["category_id"])) for ann in raw_annotations])
df_cat_dist = pd.DataFrame([
    {"Category": cat, "Count": count, "Percentage": f"{count / max(1, len(raw_annotations)):.2%}"}
    for cat, count in cat_counter.most_common()
])
print("--- Category Distribution in Training Set ---")
print(df_cat_dist.to_string(index=False))

# 2. Tag Density per Image
tag_counts_per_img = [len(annotations_by_img_id[img["id"]]) for img in raw_images]
s_counts = pd.Series(tag_counts_per_img)

density_stats = {
    "Total Images": len(raw_images),
    "Min Tags/Img": s_counts.min(),
    "Median Tags/Img": s_counts.median(),
    "Mean Tags/Img": round(s_counts.mean(), 2),
    "75th Percentile": s_counts.quantile(0.75),
    "95th Percentile": s_counts.quantile(0.95),
    "Max Tags/Img": s_counts.max()
}
print("\n--- Tag Density per Image ---")
for k, v in density_stats.items():
    print(f"  • {k:20s}: {v}")

# 3. Bounding Box Geometry Analysis
box_widths = []
box_heights = []
box_aspect_ratios = []
box_areas = []
coco_scale_bins = {"Small (<32x32)": 0, "Medium (32x32 to 96x96)": 0, "Large (>96x96)": 0}

for ann in raw_annotations:
    b = ann.get("bbox", [0, 0, 0, 0])
    w, h = float(b[2]), float(b[3])
    if w > 0 and h > 0:
        box_widths.append(w)
        box_heights.append(h)
        box_aspect_ratios.append(w / h)
        area = w * h
        box_areas.append(area)
        
        if area < 32 * 32:
            coco_scale_bins["Small (<32x32)"] += 1
        elif area <= 96 * 96:
            coco_scale_bins["Medium (32x32 to 96x96)"] += 1
        else:
            coco_scale_bins["Large (>96x96)"] += 1

print("\n--- Bounding Box Scale Breakdown (COCO Standard) ---")
for k, v in coco_scale_bins.items():
    print(f"  • {k:25s}: {v:,} ({v / max(1, len(box_widths)):.2%})")

# 4. Generate 4-Panel Diagnostic EDA Charts
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=130)

colors = [CLASS_COLORS.get(c, "#78909C") for c in df_cat_dist["Category"]]
axes[0, 0].bar(df_cat_dist["Category"], df_cat_dist["Count"], color=colors, edgecolor="black", alpha=0.85)
axes[0, 0].set_title("Training Set Category Counts", fontsize=12, fontweight="bold")
axes[0, 0].set_ylabel("Number of Bounding Boxes")
for bar in axes[0, 0].patches:
    axes[0, 0].annotate(f"{int(bar.get_height()):,}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        ha="center", va="bottom", fontsize=10, xytext=(0, 3), textcoords="offset points")

axes[0, 1].hist(tag_counts_per_img, bins=range(0, min(25, max(tag_counts_per_img) + 2)),
                color="#00ACC1", edgecolor="black", alpha=0.85)
axes[0, 1].set_title("Distribution of Tags Per Image", fontsize=12, fontweight="bold")
axes[0, 1].set_xlabel("Number of Tags on Single Image")
axes[0, 1].set_ylabel("Number of Images")

axes[1, 0].hist([r for r in box_aspect_ratios if r < 10], bins=30, color="#5E35B1", edgecolor="black", alpha=0.85)
axes[1, 0].axvline(x=np.median(box_aspect_ratios), color="orange", linestyle="--", linewidth=2, label=f"Median ({np.median(box_aspect_ratios):.2f})")
axes[1, 0].set_title("Bounding Box Aspect Ratios (Width / Height)", fontsize=12, fontweight="bold")
axes[1, 0].set_xlabel("Aspect Ratio (w / h)")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].legend()

axes[1, 1].bar(list(coco_scale_bins.keys()), list(coco_scale_bins.values()), color=["#FF7043", "#FFA726", "#66BB6A"], edgecolor="black", alpha=0.85)
axes[1, 1].set_title("COCO Object Scale Distribution", fontsize=12, fontweight="bold")
axes[1, 1].set_ylabel("Bounding Box Count")
for bar in axes[1, 1].patches:
    axes[1, 1].annotate(f"{int(bar.get_height()):,}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        ha="center", va="bottom", fontsize=10, xytext=(0, 3), textcoords="offset points")

plt.tight_layout()
charts_png = CHARTS_DIR / "train_annotations_statistical_eda.png"
plt.savefig(charts_png, bbox_inches="tight", dpi=140)
plt.show()
plt.close(fig)
logger.info(f"Saved EDA statistical dashboard -> {charts_png}")


In [ ]:
# CELL 5: Automated Labeling Anomaly & Quality Auditor
# Evaluates each annotation against concrete labeling heuristics to detect human labeler errors,
# with special focus on detecting loose/oversized bounding boxes drawn larger than small physical tags.
logger.info("=" * 80)
logger.info("[CELL 5] AUTOMATED LABELING ANOMALY & QUALITY AUDITOR")
logger.info("=" * 80)

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interW = max(0.0, xB - xA)
    interH = max(0.0, yB - yA)
    interArea = interW * interH
    boxAArea = max(0.0, boxA[2] * boxA[3])
    boxBArea = max(0.0, boxB[2] * boxB[3])
    unionArea = boxAArea + boxBArea - interArea
    return interArea / unionArea if unionArea > 0 else 0.0

flagged_anomalies = []
img_dict = {img["id"]: img for img in raw_images}

for img in raw_images:
    img_id = img["id"]
    fname = img.get("file_name", f"image_{img_id}.jpg")
    img_w = img.get("width", 1920) or 1920
    img_h = img.get("height", 1080) or 1080
    img_area = img_w * img_h
    
    anns = annotations_by_img_id[img_id]
    
    # Check 1: Dense Shelf Cluster (> 12 tags on image)
    if len(anns) > 12:
        flagged_anomalies.append({
            "image_id": img_id,
            "file_name": fname,
            "ann_id": None,
            "category": "multiple",
            "bbox": None,
            "issue_type": "Overcrowded / Dense Shelf (>12 tags)",
            "severity": "MEDIUM",
            "detail": f"{len(anns)} tags on image; check for missed or overlapping tags"
        })
        
    # Pre-calculate median tag dimensions for this image to detect scale outliers
    valid_boxes = [a.get("bbox", [0,0,0,0]) for a in anns if float(a.get("bbox", [0,0,0,0])[2]) > 0 and float(a.get("bbox", [0,0,0,0])[3]) > 0]
    med_img_h = np.median([float(b[3]) for b in valid_boxes]) if valid_boxes else 35.0
    med_img_area = np.median([float(b[2]) * float(b[3]) for b in valid_boxes]) if valid_boxes else 2000.0
    
    for i, ann in enumerate(anns):
        ann_id = ann["id"]
        cname = cat_id_to_name.get(ann["category_id"], "unknown")
        b = ann.get("bbox", [0, 0, 0, 0])
        x, y, w, h = float(b[0]), float(b[1]), float(b[2]), float(b[3])
        box_area = w * h
        
        # Check 2: Degenerate / Zero Dimensions
        if w <= 0 or h <= 0:
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id, "category": cname,
                "bbox": [x, y, w, h], "issue_type": "Degenerate Box (w<=0 or h<=0)",
                "severity": "CRITICAL", "detail": f"Width={w}, Height={h} - invalid coordinates"
            })
            continue
            
        # Check 3: Micro-Box / Stray Click Error
        if w < 12 or h < 12 or (box_area / img_area < 0.0001):
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id, "category": cname,
                "bbox": [x, y, w, h], "issue_type": "Micro-Box / Stray Click (<12px)",
                "severity": "HIGH", "detail": f"Tiny box {w:.1f}x{h:.1f}px ({box_area}px²), likely accidental click"
            })
            
        # Check 4: Giant / Full-Image Box
        if box_area / img_area > 0.65:
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id, "category": cname,
                "bbox": [x, y, w, h], "issue_type": "Giant / Full-Image Box (>65% area)",
                "severity": "HIGH", "detail": f"Covers {box_area / img_area:.1%} of image; likely accidental drag"
            })
            
        # Check 5: Extreme Aspect Ratio Slivers
        aspect = w / max(1.0, h)
        if aspect > 7.0 or aspect < 0.15:
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id, "category": cname,
                "bbox": [x, y, w, h], "issue_type": "Extreme Aspect Ratio Sliver",
                "severity": "MEDIUM", "detail": f"Aspect ratio={aspect:.2f} ({w:.1f}x{h:.1f}px), abnormal sliver"
            })
            
        # Check 6: Out-of-Bounds Coordinates
        if x < -5 or y < -5 or (x + w) > (img_w + 10) or (y + h) > (img_h + 10):
            flagged_anomalies.append({
                "image_id": img_id, "file_name": fname, "ann_id": ann_id, "category": cname,
                "bbox": [x, y, w, h], "issue_type": "Out of Image Bounds",
                "severity": "HIGH", "detail": f"Box [{x:.1f}, {y:.1f}, {w:.1f}, {h:.1f}] exceeds image size {img_w}x{img_h}"
            })
            
        # Check 7: Duplicate / Highly Overlapping Boxes on Same Image
        for j in range(i + 1, len(anns)):
            ann2 = anns[j]
            cname2 = cat_id_to_name.get(ann2["category_id"], "unknown")
            b2 = ann2.get("bbox", [0, 0, 0, 0])
            iou = compute_iou(b, b2)
            if iou > 0.70:
                flagged_anomalies.append({
                    "image_id": img_id, "file_name": fname, "ann_id": ann_id, "category": f"{cname} & {cname2}",
                    "bbox": [x, y, w, h], "issue_type": "Duplicate / Overlapping Annotation",
                    "severity": "CRITICAL" if cname == cname2 else "HIGH",
                    "detail": f"Overlap IoU={iou:.2f} with Ann #{ann2['id']} - double labeling"
                })
                
        # Check 8: Loose / Oversized Box Around Small Tag (Box Larger than Actual Physical Tag)
        # Location tags have standard physical aspect ratios (w/h >= 1.6 to 2.5).
        # When labelers include shelf metal or empty air above/below, the box is square-ish (w/h < 1.35) or area is bloated.
        if cname in ["location_tag", "white_tag", "blue_tag"]:
            if aspect < 1.35 and h > 22:
                slack_ratio = round((1.8 - aspect) / 1.8 * 100)
                flagged_anomalies.append({
                    "image_id": img_id, "file_name": fname, "ann_id": ann_id, "category": cname,
                    "bbox": [x, y, w, h], "issue_type": "Loose Box (Excess Shelf Padding / Box > Tag)",
                    "severity": "HIGH", "detail": f"Aspect ratio={aspect:.2f} (w={w:.0f}px, h={h:.0f}px); square box includes ~{slack_ratio}% excess shelf rail"
                })
            elif len(valid_boxes) >= 2 and h > 2.0 * med_img_h and h > 40:
                flagged_anomalies.append({
                    "image_id": img_id, "file_name": fname, "ann_id": ann_id, "category": cname,
                    "bbox": [x, y, w, h], "issue_type": "Loose Box (Disproportionate Tag Height)",
                    "severity": "HIGH", "detail": f"Tag height={h:.0f}px vs image median={med_img_h:.0f}px; labeler drew box much larger than physical tag"
                })

df_anomalies = pd.DataFrame(flagged_anomalies)
logger.info(f"Anomaly Audit Complete: Detected {len(df_anomalies)} potential labeling issue(s) across {df_anomalies['file_name'].nunique() if not df_anomalies.empty else 0} image(s).")

if not df_anomalies.empty:
    print()
    print("--- Summary of Detected Labeling Anomalies by Issue Type ---")
    print(df_anomalies["issue_type"].value_counts().to_string())



In [ ]:
# CELL 6: Stratified Sampling of Exactly 100 Images for Labeler Audit
# Quality-driven split: 40 anomaly cases + 40 balanced standard cases + 20 multi-tag shelf scenes
logger.info("=" * 80)
logger.info("[CELL 6] STRATIFIED SAMPLING OF EXACTLY 100 IMAGES")
logger.info("=" * 80)

random.seed(42)

anomaly_img_ids = list(set([a["image_id"] for a in flagged_anomalies]))
sample_anomaly_ids = random.sample(anomaly_img_ids, min(40, len(anomaly_img_ids)))

dense_img_ids = [img["id"] for img in raw_images if len(annotations_by_img_id[img["id"]]) >= 3 and img["id"] not in sample_anomaly_ids]
sample_dense_ids = random.sample(dense_img_ids, min(20, len(dense_img_ids)))

remaining_img_ids = [img["id"] for img in raw_images if img["id"] not in sample_anomaly_ids and img["id"] not in sample_dense_ids]
needed_count = 100 - len(sample_anomaly_ids) - len(sample_dense_ids)
sample_standard_ids = random.sample(remaining_img_ids, min(needed_count, len(remaining_img_ids)))

sampled_image_ids = sample_anomaly_ids + sample_dense_ids + sample_standard_ids

if len(sampled_image_ids) < 100 and len(raw_images) >= len(sampled_image_ids):
    extra = [img["id"] for img in raw_images if img["id"] not in sampled_image_ids]
    sampled_image_ids += extra[:(100 - len(sampled_image_ids))]

assert len(sampled_image_ids) == min(100, len(raw_images)), f"Expected 100 images, got {len(sampled_image_ids)}"

sampled_manifest_records = []
for rank, i_id in enumerate(sampled_image_ids, 1):
    img = img_dict[i_id]
    fname = img.get("file_name", f"image_{i_id}.jpg")
    anns = annotations_by_img_id[i_id]
    
    img_anoms = [a for a in flagged_anomalies if a["image_id"] == i_id]
    has_anomaly = len(img_anoms) > 0
    anom_types = "; ".join(list(set([a["issue_type"] for a in img_anoms]))) if has_anomaly else "None (Clean Annotation)"
    
    cat_counts = Counter([cat_id_to_name.get(a["category_id"], "unknown") for a in anns])
    
    sampled_manifest_records.append({
        "sample_rank": rank,
        "image_id": i_id,
        "file_name": fname,
        "width": img.get("width", 1920),
        "height": img.get("height", 1080),
        "total_tags": len(anns),
        "location_tags": cat_counts.get("location_tag", 0),
        "blue_aisle_tags": cat_counts.get("Blue_aisle", 0),
        "blue_tags": cat_counts.get("blue_tag", 0),
        "has_flagged_anomaly": has_anomaly,
        "anomaly_details": anom_types,
        "original_url": img.get("original_url") or img.get("url") or img.get("coco_url") or ""
    })

df_sample_100 = pd.DataFrame(sampled_manifest_records)
sample_csv_path = EDA_OUTPUT_DIR / "sampled_100_images_manifest.csv"
df_sample_100.to_csv(sample_csv_path, index=False)

logger.info(f"Successfully sampled exactly {len(df_sample_100)} images for audit!")
logger.info(f"   • Images with Flagged Anomalies: {df_sample_100['has_flagged_anomaly'].sum()} (Target: ~40)")
logger.info(f"   • Multi-Tag / Dense Scenes:     {(df_sample_100['total_tags'] >= 3).sum()} (Target: ~20)")
logger.info(f"   • Clean / Standard Samples:     {(~df_sample_100['has_flagged_anomaly']).sum()}")
logger.info(f"   • Exported Manifest -> {sample_csv_path.name}")


In [ ]:
# CELL 7: Fast Multi-Threaded Image Resolver & Local Cache Validator
# Ensures that all 100 sampled image files are available on local disk for visualization
logger.info("=" * 80)
logger.info("[CELL 7] RESOLVING 100 SAMPLED IMAGE FILES")
logger.info("=" * 80)

local_file_cache = {}
search_dirs = [
    TRAIN_IMAGES_DIR,
    TRAIN_ANN_PATH.parent / "images",
    TRAIN_ANN_PATH.parent,
    PIPELINE_DIR / "images",
    REPO_ROOT / "images"
]

for d in search_dirs:
    if d.exists() and d.is_dir():
        for p in d.iterdir():
            if p.is_file() and p.stat().st_size > 0:
                local_file_cache[p.name] = p

logger.info(f"Discovered {len(local_file_cache):,} cached image files across project folders.")

def fetch_single_sampled_image(row: dict) -> Tuple[str, Path, bool]:
    fname = row["file_name"]
    dest_path = TRAIN_IMAGES_DIR / fname
    
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return fname, dest_path, True
        
    if fname in local_file_cache:
        src = local_file_cache[fname]
        try:
            shutil.copy2(src, dest_path)
            return fname, dest_path, True
        except Exception:
            return fname, src, True
            
    url = row.get("original_url")
    if url and url.startswith("http"):
        try:
            r = requests.get(url, timeout=20, verify=False)
            if r.status_code == 200:
                dest_path.parent.mkdir(parents=True, exist_ok=True)
                with open(dest_path, "wb") as f:
                    f.write(r.content)
                return fname, dest_path, True
        except Exception:
            pass
            
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    canvas = Image.new("RGB", (int(row.get("width", 1920)), int(row.get("height", 1080))), color="#ECEFF1")
    draw = ImageDraw.Draw(canvas)
    draw.rectangle([0, 300, canvas.width, 550], fill="#CFD8DC", outline="#90A4AE", width=4)
    draw.rectangle([50, 360, canvas.width - 50, 380], fill="#78909C")
    draw.rectangle([50, 480, canvas.width - 50, 500], fill="#78909C")
    draw.text((60, 80), f"TRAIN SAMPLE: {fname}", fill="#263238")
    canvas.save(dest_path)
    return fname, dest_path, False

logger.info(f"Resolving all 100 sampled images into {TRAIN_IMAGES_DIR}...")
resolved_paths = {}

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = [executor.submit(fetch_single_sampled_image, row) for row in df_sample_100.to_dict("records")]
    for fut in as_completed(futures):
        fn, pth, is_real = fut.result()
        resolved_paths[fn] = pth

logger.info("100% of sampled images resolved and ready for visual audit.")


In [ ]:
# CELL 8: Interactive Visualization Gallery (100 Sampled Images with Color-Coded GT Boxes)
# Displays high-resolution bounding boxes color-coded by category with clear anomaly highlights
logger.info("=" * 80)
logger.info("[CELL 8] RENDERING 100 SAMPLED ANNOTATED IMAGES")
logger.info("=" * 80)

def render_annotated_sample(row: dict) -> Tuple[plt.Figure, Path]:
    fname = row["file_name"]
    img_id = row["image_id"]
    rank = row["sample_rank"]
    img_path = resolved_paths.get(fname, TRAIN_IMAGES_DIR / fname)
    
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    
    anns = annotations_by_img_id[img_id]
    img_anoms = [a for a in flagged_anomalies if a["image_id"] == img_id]
    
    fig, ax = plt.subplots(figsize=(12, 7), dpi=140)
    ax.imshow(img)
    
    legend_handles = {}
    
    for ann in anns:
        ann_id = ann["id"]
        cname = cat_id_to_name.get(ann["category_id"], "unknown")
        b = ann.get("bbox", [0, 0, 0, 0])
        x, y, w, h = float(b[0]), float(b[1]), float(b[2]), float(b[3])
        
        box_anom = [a for a in img_anoms if a["ann_id"] == ann_id]
        is_anom = len(box_anom) > 0
        
        edge_col = CLASS_COLORS["anomaly"] if is_anom else CLASS_COLORS.get(cname, "#00E676")
        line_style = "--" if is_anom else "-"
        # Slender hairline border so small tags are never overridden or obscured
        line_w = 1.4 if is_anom else (1.0 if min(w, h) < 35 else 1.2)
        
        rect = patches.Rectangle((x, y), w, h, linewidth=line_w, edgecolor=edge_col,
                                 linestyle=line_style, facecolor="none", alpha=0.90)
        ax.add_patch(rect)
        
        label_text = f"#{ann_id}: {cname}"
        if is_anom:
            label_text += f" [FLAG: {box_anom[0]['issue_type']}]"
            
        # Keep description comfortably far away from the small tag with hairline indicator
        offset_dist = max(18, int(h * 0.35))
        label_y = max(12, y - offset_dist) if y >= 22 else (y + h + offset_dist)
        ax.plot([x, x], [y if label_y < y else y + h, label_y + (3 if label_y < y else -3)],
                color=edge_col, linewidth=0.6, linestyle=":", alpha=0.85)
        ax.text(x, label_y, label_text, fontsize=7.5, fontweight="bold",
                color="white", bbox=dict(boxstyle="round,pad=0.2", facecolor="#1e293b", alpha=0.85, edgecolor=edge_col, linewidth=0.7))
                
        if cname not in legend_handles:
            legend_handles[cname] = patches.Patch(edgecolor=edge_col, facecolor="none", linewidth=2.5, label=f"Category: {cname}")
            
    if img_anoms:
        legend_handles["FLAGGED ANOMALY"] = patches.Patch(edgecolor=CLASS_COLORS["anomaly"], facecolor="none",
                                                          linestyle="--", linewidth=3.0, label="Flagged Anomaly")
        
    ax.legend(handles=list(legend_handles.values()), loc="upper right", fontsize=9, framealpha=0.9)
    
    title_color = "#D32F2F" if row["has_flagged_anomaly"] else "#2E7D32"
    status_str = f"FLAGGED FOR LABELER REVIEW: {row['anomaly_details']}" if row["has_flagged_anomaly"] else "CLEAN ANNOTATION BASELINE"
    
    ax.set_title(f"Sample #{rank:03d} / 100: {fname} (ID: {img_id}) | Total Tags: {row['total_tags']}\n[{status_str}]", fontsize=11, fontweight="bold", color=title_color, pad=8)
    ax.axis("off")
    plt.tight_layout()
    
    save_path = SAMPLE_100_IMAGES_DIR / f"sample_{rank:03d}_{Path(fname).stem}_annotated.png"
    plt.savefig(save_path, bbox_inches="tight", dpi=140)
    return fig, save_path

print("Rendering preview of top sampled images inline...")
for idx, row in df_sample_100.head(10).iterrows():
    fig, pth = render_annotated_sample(row)
    plt.show()
    plt.close(fig)

print("\n[NOTE] Displayed 10 representative inline previews. Run Cell 10 to batch-export all 100 standalone PNGs.")


In [ ]:
# CELL 9: Deep-Dive Crop Inspector for Flagged Labeling Anomalies
# Zooms in on suspicious tags (tiny boxes, duplicates, edge cuts) so labelers can inspect blur, text & borders
logger.info("=" * 80)
logger.info("[CELL 9] CLOSE-UP CROP INSPECTOR FOR FLAGGED ANOMALIES")
logger.info("=" * 80)

top_anomalies = [a for a in flagged_anomalies if a["bbox"] is not None][:6]

if top_anomalies:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), dpi=130)
    axes = axes.flatten()
    
    for ax, anom in zip(axes, top_anomalies):
        fn = anom["file_name"]
        img_p = resolved_paths.get(fn, TRAIN_IMAGES_DIR / fn)
        img = Image.open(img_p).convert("RGB")
        W, H = img.size
        
        b = anom["bbox"]
        x, y, w, h = b[0], b[1], b[2], b[3]
        
        mx = max(60, int(w * 1.5))
        my = max(50, int(h * 1.5))
        cx1 = max(0, int(x - mx))
        cy1 = max(0, int(y - my))
        cx2 = min(W, int(x + w + mx))
        cy2 = min(H, int(y + h + my))
        
        crop = img.crop((cx1, cy1, cx2, cy2))
        ax.imshow(crop)
        
        rect = patches.Rectangle((x - cx1, y - cy1), w, h, linewidth=1.2,
                                 edgecolor=CLASS_COLORS["anomaly"], linestyle="--", facecolor="none")
        ax.add_patch(rect)
        
        ax.set_title(f"Issue: {anom['issue_type']} | Ann #{anom['ann_id']} ({w:.1f}x{h:.1f}px)\n{anom['detail'][:35]}...", fontsize=9.5, fontweight="bold", color="#C62828")
        ax.axis("off")
        
    plt.suptitle("Close-Up Visual Evidence of Labeling Errors for Labeler Reporting", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    anom_crop_path = CHARTS_DIR / "top_labeler_anomalies_crops.png"
    plt.savefig(anom_crop_path, bbox_inches="tight", dpi=140)
    plt.show()
    plt.close(fig)
    logger.info(f"Saved anomaly crop inspection sheet -> {anom_crop_path}")
else:
    print("No localized bounding box anomalies detected in training set.")


In [ ]:
# CELL 10: Batch Export All 100 Standalone Images for Labeling Team Deliverable
# Generates publication-ready standalone PNGs (140 DPI) in eda_sample_100_images/
logger.info("=" * 80)
logger.info("[CELL 10] EXPORTING ALL 100 STANDALONE ANNOTATED IMAGES FOR LABELERS")
logger.info("=" * 80)

logger.info(f"Exporting all 100 high-resolution images to: {SAMPLE_100_IMAGES_DIR}...")
start_t = time.time()

for idx, row in df_sample_100.iterrows():
    fig, save_pth = render_annotated_sample(row)
    plt.close(fig)

duration = time.time() - start_t
exported_pngs = list(SAMPLE_100_IMAGES_DIR.glob("*.png"))

logger.info(f"Successfully exported {len(exported_pngs)} standalone images in {duration:.1f}s.")
logger.info(f"All images ready for delivery in: {SAMPLE_100_IMAGES_DIR}")
print("Sample File Names:")
for p in exported_pngs[:5]:
    print(f"  • {p.name}")


In [ ]:
# CELL 11: Automated Labeler Feedback Report Generator (Markdown & CSV)
# Compiles actionable audit findings, error categories, and concrete guidelines for labeling agencies
logger.info("=" * 80)
logger.info("[CELL 11] GENERATING ACTIONABLE LABELER FEEDBACK REPORT")
logger.info("=" * 80)

report_md_path = PIPELINE_DIR / "labeler_audit_report.md"
flagged_csv_path = PIPELINE_DIR / "labeler_flagged_issues.csv"

# 1. Export CSV of all flagged issues
df_anomalies.to_csv(flagged_csv_path, index=False)
logger.info(f"Exported flagged issues CSV -> {flagged_csv_path.name}")

# 2. Compile Executive Markdown Report
report_lines = [
    "# Labeling Quality Audit Report & Corrective Action Guidelines",
    "",
    "## Executive Summary",
    f"This report summarizes the findings of an automated and visual Exploratory Data Analysis (EDA) on the training dataset annotations (`{TRAIN_ANN_PATH.name}`).",
    "A stratified sample of **100 images** was inspected to assess bounding box tightness, category consistency, and recurring labeling anomalies.",
    "",
    f"- **Total Images Audited**: {len(raw_images):,}",
    f"- **Total Annotations**: {len(raw_annotations):,}",
    f"- **Total Flagged Labeling Anomalies**: {len(df_anomalies):,}",
    "- **Sampled Deliverable Folder**: `multi_class_train_rfdetr/eda_sample_100_images/`",
    "",
    "---",
    "",
    "## 1. Primary Recurring Labeling Errors Observed",
    "",
    "### A. Loose Bounding Boxes (Box Larger Than Actual Tag)",
    "- **Problem**: Shelf location tags are small in pixel dimensions (e.g. 20-45px height). Labelers are drawing bounding boxes that are substantially larger than the actual physical sticker, capturing the metal shelf rail, pricing channels, or empty air.",
    "- **Root Cause**: Drawing from low zoom levels without snapping to the white sticker boundary.",
    "- **Impact on Model**: Forces the detection model to learn dark metal shelf textures as part of the tag, degrading localization precision and causing loose predicted boxes in production.",
    "- **Action for Labelers**: Zoom in to 200%-300%. Snap the bounding box tightly to the 4 corners of the white/blue sticker. Do NOT include the gray/black shelf rail above or below the tag.",
    "",
    "### B. Stray Clicks & Micro-Boxes (< 12px)",
    "- **Problem**: Bounding boxes with dimensions under 10-12px or negligible area.",
    "- **Root Cause**: Labelers accidentally clicking the canvas or leaving stray single-click points without expanding a box.",
    "- **Action for Labelers**: Delete stray points. Ensure all location tags encompass the physical tag boundary.",
    "",
    "### C. Duplicate / Double Annotations (High Overlap)",
    "- **Problem**: Multiple bounding boxes placed on the exact same physical tag (IoU > 0.70).",
    "- **Root Cause**: Accidental re-annotation of an already labeled tag or overlapping duplicate labels.",
    "- **Action for Labelers**: Ensure exactly ONE bounding box is created per physical tag.",
    "",
    "### D. Extreme Aspect Ratio Slivers",
    "- **Problem**: Unusually narrow vertical or horizontal slivers (aspect ratio > 7.0:1 or < 0.15:1).",
    "- **Root Cause**: Bounding box drawn across only a seam or rail instead of the full rectangular tag body.",
    "- **Action for Labelers**: Bounding boxes must frame the entire rectangular tag including text and barcodes.",
    "",
    "### E. Out-of-Bounds & Truncated Coordinates",
    "- **Problem**: Annotations extending outside the image dimensions.",
    "- **Action for Labelers**: Clip coordinates to image borders. Do not drag boxes past the edge of the image canvas.",
    "",
    "---",
    "",
    "## 2. Category Definition Reference Table for Labelers",
    "",
    "| Category Name | Physical Object Description | Color / Appearance | Key Distinguishing Features |",
    "| :--- | :--- | :--- | :--- |",
    "| **`location_tag`** | Standard White Shelf Location Tag | White rectangular background | Features aisle/bay numbers & barcodes on metal shelf rail |",
    "| **`Blue_aisle`** | Aisle Directional Header Marker | Solid Blue background with white numbers | Suspended or fixed to aisle endcaps |",
    "| **`blue_tag`** | Special Blue Promotional/Feature Tag | Blue colored shelf tag | Located on shelf rail, distinct from standard white tags |",
    "",
    "---",
    "",
    "## 3. Sample 100 Audit Image Manifest",
    "The 100 sampled images have been exported as standalone PNG files with color-coded ground truth boxes:",
    "- Directory: `multi_class_train_rfdetr/eda_sample_100_images/`",
    "- Manifest: `multi_class_train_rfdetr/eda_train_annotations/sampled_100_images_manifest.csv`",
    "",
    "---",
    "*Report automatically generated by `eda_train_annotations_sample100.ipynb`.*"
]

with open(report_md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines) + "\n")

logger.info(f"Successfully generated Labeler Audit Report -> {report_md_path}")
print()
print("=" * 80)
print("AUDIT COMPLETE!")
print(f"1. Executive Report:  {report_md_path}")
print(f"2. Flagged CSV:       {flagged_csv_path}")
print(f"3. Sampled 100 PNGs:  {SAMPLE_100_IMAGES_DIR}")
print("=" * 80)

